In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
import sys
import os

# Agregar el directorio padre al path para poder importar módulos propios
sys.path.append(os.path.abspath('..'))

# Importar el modelo de árbol de decisión personalizado
from src.tree_models import DecisionTreeClassifierCustom

# Crear un conjunto de datos de juguete (toy dataset)
data = {
    'Clima': ['Sol', 'Sol', 'Nublado', 'Lluvia', 'Lluvia', 'Lluvia', 'Nublado', 'Sol', 'Sol', 'Lluvia', 'Sol', 'Nublado', 'Nublado', 'Lluvia'],
    'Temperatura': [85, 80, 83, 70, 68, 65, 64, 72, 69, 75, 75, 72, 81, 71],
    'Viento': ['Suave', 'Fuerte', 'Suave', 'Suave', 'Suave', 'Fuerte', 'Fuerte', 'Suave', 'Suave', 'Suave', 'Fuerte', 'Fuerte', 'Suave', 'Fuerte'],
    'Resultado': ['L', 'L', 'E', 'V', 'V', 'L', 'E', 'L', 'V', 'V', 'E', 'E', 'E', 'L'] 
}
df_toy = pd.DataFrame(data)

# Separar las variables predictoras (X) de la variable objetivo (y)
X_toy = df_toy.drop('Resultado', axis=1)
y_toy = df_toy['Resultado']

# Inicializar y entrenar el modelo de árbol con una ganancia de información mínima
tree_model = DecisionTreeClassifierCustom(min_info_gain=0.01)
tree_model.fit(X_toy, y_toy)

# Realizar predicciones y evaluar la precisión del modelo
preds = tree_model.predict(X_toy)
print(f"Predicciones: {preds}")
print(f"Precisión en el entrenamiento: {accuracy_score(y_toy, preds)}")


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit 

from src.pipeline import get_train_test_data, preprocessing_pipeline
from src.tree_models import DecisionTreeClassifierCustom

print("Cargando y procesando los datos...")
# Cargar y dividir los datos en el tiempo (Train: ->2023 | Test: 2024-2025)
X_train, y_train, X_test, y_test = get_train_test_data(filepath='../data/raw/futbol_uruguayo.csv')

# Aplicar el pipeline de preprocesamiento del Rol 1
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)
nomi_colonne = preprocessing_pipeline.named_steps['encoder'].get_feature_names_out()

# Convertir a DataFrame 
X_train_visibile = pd.DataFrame(X_train_processed, columns=nomi_colonne)
X_test_visibile = pd.DataFrame(X_test_processed, columns=nomi_colonne)

print("Buscando los mejores hiperparámetros para el Árbol Personalizado...")

# Definir la validación cruzada temporal (Divide el pasado en 5 bloques cronológicos)
tscv = TimeSeriesSplit(n_splits=5)

# Definir la grilla de valores que queremos probar
param_grid_arvore = {
    'min_info_gain': [0.0, 0.0005, 0.001, 0.005, 0.01, 0.05]
}

# Configurar la búsqueda automatizada (GridSearchCV)
grid_arvore = GridSearchCV(
    estimator=DecisionTreeClassifierCustom(),
    param_grid=param_grid_arvore,
    cv=tscv,
    scoring='f1_macro', 
    return_train_score=True
)

# Iniciar el entrenamiento iterativo (ESTO PUEDE TARDAR UNOS MINUTOS)
grid_arvore.fit(X_train_visibile, y_train)

# Mostrar el mejor resultado encontrado durante la validación
mejor_ganancia = grid_arvore.best_params_['min_info_gain']
print(f"Búsqueda terminada! Mejor min_info_gain encontrado: {mejor_ganancia}")

print("Realizando predicciones para 2024-2025 con el MEJOR modelo...")
# Extraer automáticamente la mejor versión del árbol
arvore_mejor_modelo = grid_arvore.best_estimator_

# Evaluando estrictamente en la ventana temporal de prueba
preds_arvore = arvore_mejor_modelo.predict(X_test_visibile)

# Calculando las métricas exigidas por la tarea
macro_f1 = f1_score(y_test, preds_arvore, average='macro')
acuracia = accuracy_score(y_test, preds_arvore)

print(f"\nResultados del Mejor Árbol de Decisión:")
print(f"Precisión (Accuracy): {acuracia:.4f}")
print(f"Macro-F1 (Métrica Principal): {macro_f1:.4f}")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, preds_arvore, labels=['L', 'E', 'V']))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

print("Buscando los mejores hiperparametros para el Random Forest...")

# 1. Validacion cruzada temporal
tscv = TimeSeriesSplit(n_splits=5)

# 2. Grilla de hiperparametros
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None]
}

# 3. Busqueda automatizada 
# Aqui aplicamos el cambio de optimizacion: scoring='f1_macro'
grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1, 
    return_train_score=True
)

# 4. Entrenamiento iterativo
grid_rf.fit(X_train_visibile, y_train)

mejores_params = grid_rf.best_params_
print(f"Busqueda terminada. Mejores parametros encontrados: {mejores_params}")

print("Realizando predicciones para 2024-2025 con el mejor Random Forest...")
mejor_rf = grid_rf.best_estimator_
preds_rf = mejor_rf.predict(X_test_visibile)

# Calculo de metricas
macro_f1_rf = f1_score(y_test, preds_rf, average='macro')
acuracia_rf = accuracy_score(y_test, preds_rf)

print("\nResultados del Mejor Random Forest:")
print(f"Precision (Accuracy): {acuracia_rf:.4f}")
print(f"Macro-F1 (Metrica Principal): {macro_f1_rf:.4f}")

print("\nMatriz de Confusion:")
print(confusion_matrix(y_test, preds_rf, labels=['L', 'E', 'V']))

print("\nReporte detallado por clase (Precision, Recall, F1):")
print(classification_report(y_test, preds_rf, labels=['L', 'E', 'V']))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Extraer el diccionario de resultados generados por GridSearchCV
resultados = grid_rf.cv_results_

# 2. Filtrar para mantener constante el mejor n_estimators (200)
# Esto nos permite ver el efecto aislado de max_depth
indices_optimos = [i for i, params in enumerate(resultados['params']) if params['n_estimators'] == 200]

# 3. Extraer los valores de max_depth (convirtiendo 'None' a texto para poder graficarlo)
profundidades = [str(resultados['params'][i]['max_depth']) for i in indices_optimos]

# 4. Calcular el error (1 - puntuacion) para entrenamiento y validacion
error_entrenamiento = 1 - resultados['mean_train_score'][indices_optimos]
error_validacion = 1 - resultados['mean_test_score'][indices_optimos]

# 5. Construccion del grafico
plt.figure(figsize=(10, 6))
plt.plot(profundidades, error_entrenamiento, marker='o', label='Error de Entrenamiento (Train)', color='blue')
plt.plot(profundidades, error_validacion, marker='s', label='Error de Validacion (Test CV)', color='red')

# 6. Etiquetas y formato para el informe IEEE
plt.title('Error de Clasificacion vs Profundidad Maxima del Arbol (Random Forest con 200 arboles)')
plt.xlabel('Profundidad Maxima (max_depth)')
plt.ylabel('Error (1 - Macro-F1)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.show()